# Неделя 4 — Итоговый отчёт

Задание: `docs/week4_experiment.md`

## 1. Постановка задачи и определение оттока  
Цель задачи - заранее находить клиентов с высоким риском оттока, чтобы компания успела связаться с ними и предложить варианты удержания  
Отток - Явление при котором у клиента средняя выручка составляет менее 20% от средней выручки за шестимесячное окно наблюдения. Между окнами наблюдение используется зазор в два месяца.

## 2. Данные: объём, период, доля оттока по сегментам

В исходных данных 50 400 клиентов и 1 073 079 строк помесячного использования за период январь-2024 по июнь 2026

Итоговый датасет содержит 257 631 строк и 48 696 уникальных клиентов

На финальном тесте доля оттока составляет 13.79%. По сегментам: 

- Малый бизнес - 16.77%
- средний - 12.80%
- крупный бизнес - 8.04%

Самая высокая доля оттока в малом бизнесе

## 3. Схема сборки выборки (snapshot'ы)

Для каждого snapshot используется: 

- 6 месяцев наблюдения
- 2 месяца зазора
- 3 месяца прогноза для определения таргета

Разбиение по времени: 
- train - snapshot 2024-07 и 2024-10
- valid - 2025-01
- 2025-04, 2025-07, 2025-10 не испольлзуется
- test - 2026-01

## 4. Признаки: группы и интуиция

Использовались следующие группы признаков:

- платежи: средняя, минимальная, максимальная и последняя выручка, отношение последней выручки к средней;
- динамика: тренд и волатильность выручки, количество месяцев снижения, тренд трафика;
- объём отношений: количество SIM-карт и его изменение, количество месяцев истории;
- сигналы проблем: обращения в поддержку и задолженность;
- категории: сегмент бизнеса, продукт и регион.

Основная идея — найти сокращение использования услуг и признаки проблем перед уходом клиента.

## 5. Схема валидации и метрики

Модели обучались на прошлых периодах, настраивались на valid и один раз проверялись на финальном test.

Использовались ROC-AUC, PR-AUC, Precision@10% и Lift@10%

## 6. Результаты моделей: правило → логрег → CatBoost

На выборке valid получены результаты:

| Модель | ROC-AUC | PR-AUC | Lift@10% |
|---|---:|---:|---:|
| Правило падения выручки | 0,8618 | 0,6538 | 6,93x |
| Логистическая регрессия | 0,8786 | 0,7117 | 6,97x |
| CatBoost | 0,8932 | 0,7854 | 7,48x |

CatBoost показал лучший результат и был выбран как финальная модель.

На финальном test CatBoost получил ROC-AUC = 0,9143, PR-AUC = 0,8212 и Lift@10% = 6,65x.

## 7. Эксперимент A/B/C: ответ на главный вопрос проекта

Сравнивались три подхода:

- A — единая модель без сегмента и продукта;
- B — единая модель с сегментом и продуктом;
- C — отдельная модель для каждого сегмента.

| Подход | ROC-AUC | PR-AUC | Lift@10% |
|---|---:|---:|---:|
| A | 0,8936 | 0,7827 | 7,49x |
| B | 0,8932 | 0,7854 | 7,48x |
| C | 0,8614 | 0,7304 | 6,91x |

Различие между A и B незначительное 95% 

Подход C значимо хуже B: разница AUC составляет 0,0317

Рекомендуется использовать единую модель B. Отдельные модели по сегментам не нужны.

## 8. Интерпретация: топ-факторы оттока, примеры клиентов

Главные факторы риска по SHAP — падение последней выручки относительно среднего уровня, длительное снижение выручки, сокращение количества SIM-карт, задолженность и обращения в поддержку.

Это соответствует бизнес-логике: перед уходом клиент сокращает использование услуг и чаще сталкивается с проблемами.

Для клиентов CL003097 и CL010385 модель спрогнозировала риск около 99,7%. Оба клиента действительно ушли.

## 9. Бизнес-эффект: lift, эффект от обзвона топ-10%
На финальном тесте среди топ-10% клиентов по риску 91,7% действительно уходят.

В эти 10% попадает около 66,5% всех будущих уходов. Lift = 6,65x, то есть отбор клиентов моделью в 6,65 раза эффективнее случайного выбора.

Из 40 265 тестовых клиентов отделу удержания достаточно обработать около 4 026 клиентов, чтобы найти примерно две трети будущих уходов.

## 10. Ограничения и что делать дальше
Ограничения проекта:

- таргет основан на снижении выручки и является приближённым определением оттока;
- доля оттока со временем растёт, поэтому качество модели необходимо регулярно контролировать;
- в крупном бизнесе меньше примеров оттока;
- модель проверена только на одном финальном периоде;
- не учитывались стоимость контакта и реальный эффект мер удержания.

Дальше рекомендуется провести пилотную кампанию удержания, оценить экономический эффект, настроить мониторинг качества и регулярно переобучать модель на новых данных.
